# Teesri Shikayat narration: the survey line (sentences 45–47)

Same model, settings and seeds as `narration-chatterbox.ipynb` (seed = n for take a, 1000 + n for take b), so the voice matches the other 44 clips. Runtime → T4 GPU, then Run all.


In [ ]:
!pip -q install chatterbox-tts==0.1.7 && pip -q uninstall -y torchvision  # Colab's torchvision is built for a newer torch and breaks transformers' import

In [ ]:
import json, os, zipfile, torch, torchaudio as ta
from importlib.metadata import version
from chatterbox.tts import ChatterboxTTS
from google.colab import files
LINES = json.loads(r'''[{"n": 45, "say": "So I asked eleven people in six cities."}, {"n": 46, "say": "Most never knew their water was dirty."}, {"n": 47, "say": "One friend ended up in hospital after a sewage leak."}]''')
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then Run all again"
model = ChatterboxTTS.from_pretrained(device="cuda")
os.makedirs("vo", exist_ok=True)
manifest = {"chatterbox": version("chatterbox-tts"), "torch": torch.__version__, "sr": model.sr,
            "settings": {"exaggeration": 0.4, "cfg_weight": 0.45, "temperature": 0.7}, "takes": []}
BATCH = 11  # a zip every 11 sentences, so a dropped runtime loses at most one batch
for take, base in (("a", 0), ("b", 1000)):  # every take a first: a late drop still leaves one take of each sentence
    for i in range(0, len(LINES), BATCH):
        part = LINES[i:i + BATCH]
        for l in part:
            path = f"vo/{l['n']:02d}{take}.wav"
            if os.path.exists(path):  # rerun on the same runtime: keep what is made
                wav, _ = ta.load(path)
            else:
                torch.manual_seed(base + l["n"])
                wav = model.generate(l["say"], exaggeration=0.4, cfg_weight=0.45, temperature=0.7)
                ta.save(path, wav, model.sr)
            secs = wav.shape[-1] / model.sr
            manifest["takes"].append({"n": l["n"], "take": take, "seed": base + l["n"], "seconds": round(secs, 2), "chars": len(l["say"])})
            print(f"{l['n']:02d}{take} {secs:5.1f}s  {l['say'][:60]}", flush=True)
        json.dump(manifest, open("vo/manifest.json", "w"), indent=1)
        name = f"vo-{take}-{part[0]['n']:02d}-{part[-1]['n']:02d}.zip"
        with zipfile.ZipFile(name, "w") as z:
            for l in part: z.write(f"vo/{l['n']:02d}{take}.wav")
            z.write("vo/manifest.json")
        files.download(name)
        print("downloaded", name, flush=True)